<a href="https://colab.research.google.com/github/saleet-developer/Medical-RAG-Chatbot/blob/main/notebooks/colab/embedding-generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/Medical-RAG-Project'
CHUNKS_FILE = f'{PROJECT_PATH}/processed data/Chunked-Documents/all_chunks.jsonl'
EMBEDDINGS_PATH = f'{PROJECT_PATH}/processed data/Embeddings'

In [ ]:
!pip install sentence-transformers faiss-cpu tqdm

In [ ]:
import json

chunks = []

with open(CHUNKS_FILE, 'r') as f:
     for line in f:
         chunks.append(json.loads(line))

print(f'Loaded {len(chunks)} chunks')
texts = [c['text'] for c in chunks]

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using Device {device}')

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

In [ ]:
import numpy as np
from tqdm import tqdm

batch_size = 256
embeddings_list = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]
    batch_embeddings = model.encode(batch_texts, show_progress_bar=True)
    embeddings_list.append(batch_embeddings)

embeddings = np.vstack(embeddings_list)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
np.save(f'{EMBEDDINGS_PATH}/embeddings.npy', embeddings)

with open(f'{EMBEDDINGS_PATH}/chunks_metadata.json', 'w') as f:
     json.dump(chunks, f)

print('Saved Embeddings & metadata')

In [ ]:
import faiss

dimension = embeddings.shape[1]

# using HNSW
index = faiss.IndexHNSWFlat(dimension, 32)
index.hnsw.efConstruction = 40
index.hnsw.efSearch = 16


In [ ]:
index.add(embeddings.astype(np.float32))
print(f'index size: {index.ntotal} vectors')

In [ ]:
faiss.write_index(index, f'{EMBEDDINGS_PATH}/FAISS_INDEX.BIN')

In [ ]:
# Testing
query = 'What are the symptoms of diabetes?'
query_emb = model.encode([query])

k = 5
distances, indices = index.search(query_emb.astype(np.float32), k)

print('Top 5 retrieved chunks:')
for idx in indices[0]:
    print(f"- {chunks[idx]['text'][:200]}...")